# Feel the input data: population, boundaries, missing values, validation labels

Runs **locally** from the repo `.venv` (no Earth Engine login, no network): it only reads the files already cached
under `data/`. Each section takes one cleaning rule, shows the data **before**, calls the **real pipeline code**
that applies the rule, and shows **after**.

| # | Rule | Where it lives | Status |
|---|------|----------------|--------|
| 1 | Population: drop total/subtotal rows, `"-"` -> 0, force numeric | `_clean_population_table` in [pillars.py](../src/priority_score/pillars.py) | Done, with one gap (see section 1) |
| 2 | URA boundaries: rename GeoJSON fields for Earth Engine | `sanitize_dotted_properties` in [geo.py](../src/utils/geo.py), called from [subzones.py](../src/ingest/subzones.py) | Done |
| 3 | Missing data: keep NaN, drop only when required | `zonal_mean` in [geo.py](../src/utils/geo.py) plus targeted `dropna` calls | Done |
| 4 | Validation labels: exclude blank/uncertain and NoData, count them | `evaluate_classifier` in [classifier_evaluation.py](../validation/landcover_validation/classifier_evaluation.py) | Done, but the count is printed, not saved |

The `data/` folders are git-ignored, so on a fresh clone you first need the pipeline outputs (see `SETUP.md`).

In [1]:
import copy
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "config").exists():  # opened from notebooks/ -> go up to the repo root
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config.settings import (
    BUCKET_NAMES, ELDERLY_AGE_COLUMNS, INTERIM_DIR, PROCESSED_DIR,
    SINGSTAT_NAME_COLUMN, SUBZONE_ID_PROPERTY, TOTAL_POP_COLUMN,
)
from src.ingest.singstat import fetch_population_by_subzone
from src.ingest.subzones import as_geodataframe, fetch_subzones_geojson
from src.priority_score.pillars import _clean_population_table
from src.utils.geo import sanitize_dotted_properties
from validation.landcover_validation.classifier_evaluation import sample_raster_at_points

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

BLUE, ORANGE, MUTED, LAND = "#2a78d6", "#eb6834", "#c9c8c3", "#f0efec"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlelocation": "left", "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6, "axes.axisbelow": True,
})

SENS_PATH = INTERIM_DIR / "sensitivity_pillar.csv"
LABELS_PATH = INTERIM_DIR / "validation_sample" / "validation_sample_300_labeled.csv"
print("repo root:", ROOT)

ModuleNotFoundError: No module named 'config'

## 1. Population data (SingStat)

Rule: remove total/subtotal rows, convert suppressed `"-"` to 0, enforce numeric values.

**Before.** The raw table mixes three kinds of row, and every count column is text because SingStat writes `"-"`
for suppressed cells.

In [ ]:
pop_raw = fetch_population_by_subzone()  # the cached raw table, exactly what the pipeline loads
name = pop_raw[SINGSTAT_NAME_COLUMN]
numeric_cols = [TOTAL_POP_COLUMN] + ELDERLY_AGE_COLUMNS

row_kind = np.select(
    [name == "Total", name.str.contains(" - Total", regex=False)],
    ["Singapore grand total", "planning-area subtotal"], default="subzone",
)
print(f"{pop_raw.shape[0]} rows x {pop_raw.shape[1]} columns")
print(pd.Series(row_kind).value_counts().to_string(), "\n")
pop_raw[[SINGSTAT_NAME_COLUMN, TOTAL_POP_COLUMN, "Total_65_69", "Total_70_74", "Total_90andOver"]].head(8)

In [ ]:
is_subzone = row_kind == "subzone"
suppressed = pop_raw.loc[is_subzone, numeric_cols] == "-"
print(f"Subzone rows with at least one '-' in the columns the pipeline uses: {suppressed.any(axis=1).sum()} of {is_subzone.sum()}")
print(f"Subzone rows where the total itself is '-': {suppressed[TOTAL_POP_COLUMN].sum()}  (the empty subzones)")
print("dtype of Total_Total before cleaning:", pop_raw[TOTAL_POP_COLUMN].dtype)
shown = pop_raw.loc[is_subzone].loc[suppressed.any(axis=1), [SINGSTAT_NAME_COLUMN, TOTAL_POP_COLUMN, "Total_65_69", "Total_70_74"]]
shown.sample(8, random_state=0)

**After.** The pipeline's own cleaning function, unmodified:

In [ ]:
pop = _clean_population_table(pop_raw)
print("dtypes of the count columns now:", pop[numeric_cols].dtypes.astype(str).unique().tolist())
print("non-numeric leftovers (would show as NaN):", int(pop[numeric_cols].isna().sum().sum()))
print(f"rows: {len(pop_raw)} -> {len(pop)}")
pop[numeric_cols].describe().T[["count", "min", "50%", "max"]].head(6)

**A gap worth knowing about.** The subtotal filter matches the string `" - Total"` (space, dash, space). SingStat
spells one planning area `"Changi- Total"` (no space before the dash), so that subtotal row survives cleaning:

In [ ]:
pop[pop[SINGSTAT_NAME_COLUMN].str.contains("total", case=False)][[SINGSTAT_NAME_COLUMN, TOTAL_POP_COLUMN]]

It is harmless today because the next step keeps only rows whose name matches a URA subzone, and no subzone is
called "Changi- Total" (section 2 shows it is the single unmatched row). It would double-count Changi if anyone
ever summed the cleaned table directly.

In [ ]:
subzone_pop = pop[~pop[SINGSTAT_NAME_COLUMN].str.contains("total", case=False)]
zero = (subzone_pop[TOTAL_POP_COLUMN] == 0).sum()

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(subzone_pop[TOTAL_POP_COLUMN], bins=30, color=BLUE, edgecolor="white", linewidth=1)
ax.set(xlabel="Resident population per subzone", ylabel="Number of subzones")
ax.set_title(f"Most subzones are small; the largest has {subzone_pop[TOTAL_POP_COLUMN].max():,} residents")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.text(0.30, 0.80, f"{zero} of {len(subzone_pop)} subzones have 0 residents\n(shipyards, reservoirs, the airport...)",
        transform=ax.transAxes, fontsize=9, color="#52514e")
plt.show()

## 2. URA subzone boundaries

Rule: standardise GeoJSON field names so Earth Engine accepts them. Earth Engine rejects property names containing
`.`, and data.gov.sg's ArcGIS export ships `SHAPE.AREA` / `SHAPE.LEN`.

In [ ]:
geojson = fetch_subzones_geojson()  # cached copy, already sanitised at download time
gdf = as_geodataframe(geojson)
print(f"{len(gdf)} subzones, CRS {gdf.crs}, '{SUBZONE_ID_PROPERTY}' unique: {gdf[SUBZONE_ID_PROPERTY].is_unique}")
dotted = {k for f in geojson["features"] for k in f["properties"] if "." in k}
print("property names still containing '.':", dotted or "none")
gdf.drop(columns="geometry").head(4)

What the rename does, on one feature with the raw data.gov.sg field names put back:

In [ ]:
raw_style = {"type": "FeatureCollection", "features": [{
    "type": "Feature", "geometry": None,
    "properties": {"SUBZONE_N": "DEPOT ROAD", "SHAPE.AREA": 442297.5, "SHAPE.LEN": 4281.7},
}]}
print("before:", list(raw_style["features"][0]["properties"]))
sanitize_dotted_properties(raw_style)
print("after: ", list(raw_style["features"][0]["properties"]))

Joining the two sources by subzone name (upper-cased, trimmed) is how the pipeline links population to polygons:

In [ ]:
gdf["_key"] = gdf[SUBZONE_ID_PROPERTY].str.strip().str.upper()
pop["_key"] = pop[SINGSTAT_NAME_COLUMN].str.strip().str.upper()
print("polygons with no population row:", sorted(set(gdf["_key"]) - set(pop["_key"])) or "none")
print("population rows with no polygon:", sorted(set(pop["_key"]) - set(gdf["_key"])) or "none")

sens = pd.read_csv(SENS_PATH)  # the pipeline's finished sensitivity pillar
gdf = gdf.merge(sens, left_on="_key", right_on="subzone_id", how="left", validate="one_to_one")
print(f"polygons carrying pillar values: {gdf['subzone_id'].notna().sum()} / {len(gdf)}")

## 3. Missing data

Rule: keep missing zonal values as NaN and remove them only where a step cannot cope with them (clustering, the
held-out comparisons, land-change fractions). First, where does NaN actually occur in the pipeline's outputs?

In [ ]:
tables = {
    "sensitivity pillar": SENS_PATH,
    "adaptive-capacity pillar": INTERIM_DIR / "adaptive_capacity_pillar.csv",
    "heat variants (LST)": INTERIM_DIR / "heat_variants_subzone.csv",
    "hotspot features": INTERIM_DIR / "hotspot_features_subzone.csv",
    "priority score": PROCESSED_DIR / "priority_score.csv",
}
rows = []
for label, path in tables.items():
    df = pd.read_csv(path)
    nan_cols = df.isna().sum()
    rows.append({"table": label, "rows": len(df),
                 "columns with NaN": ", ".join(f"{c} ({n})" for c, n in nan_cols[nan_cols > 0].items()) or "none"})
pd.DataFrame(rows)

Only one column has gaps, `elderly_proportion`. It is 0 residents divided by 0 residents, deliberately left as NaN
rather than 0, because 0.0 would falsely read as "no elderly" instead of "nobody lives here". Check that the NaN
rows are exactly the empty subzones:

In [ ]:
empty = sens["population_total"] == 0
print("NaN elderly_proportion == zero-population subzones:", bool((sens["elderly_proportion"].isna() == empty).all()), f"({empty.sum()} subzones)")
print("sensitivity_raw for those subzones:", sens.loc[empty, "sensitivity_raw"].unique().tolist(), "(NaN is filled to 0 in the density term)")
sens.loc[empty, "subzone_id"].head(12).tolist()

In [ ]:
gdf["density_plot"] = gdf["population_density_km2"].where(gdf["population_total"] > 0)  # empty -> NaN -> grey
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
panels = [("density_plot", "Residents per km2", "Population density"),
          ("elderly_proportion", "Share aged 65+", "Elderly share")]
for ax, (col, label, title) in zip(axes, panels):
    gdf.plot(ax=ax, column=col, cmap="Blues", legend=True, linewidth=0.15, edgecolor="white",
             legend_kwds={"label": label, "shrink": 0.6},
             missing_kwds={"color": MUTED})
    ax.plot([], [], "s", color=MUTED, label="no residents (NaN)")
    ax.legend(loc="lower left", frameon=False, fontsize=8)
    ax.set_title(title); ax.set_axis_off(); ax.grid(False)
plt.tight_layout(); plt.show()

Where the pipeline does drop NaN, and why (each is a targeted `dropna`, never a blanket one):

- [cluster.py](../src/hotspots/cluster.py): K-Means / GMM cannot take NaN.
- [modis_heldout.py](../validation/input_validation/modis_heldout.py) and
  [nea_heldout.py](../validation/input_validation/nea_heldout.py): the held-out comparison needs a real reference value.
- [land_change.py](../validation/input_validation/land_change.py): a null change fraction has nothing to compare.

## 4. Validation labels (300 hand-labelled points)

Rule: exclude blank/uncertain labels and NoData points, and record how many were excluded. The evaluator applies the
same two filters to RF, U-Net and the hybrid, so the comparison is on identical rules.

In [ ]:
labels = pd.read_csv(LABELS_PATH)
labels["agreed_label"] = labels["agreed_label"].fillna("").astype(str)
is_valid_label = labels["agreed_label"].isin(BUCKET_NAMES.values())  # same test as evaluate_classifier
print(f"{len(labels)} points; {(~is_valid_label).sum()} with a blank or uncertain label")

counts = labels["agreed_label"].replace("", "(blank)").value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 2.8))
colors = [BLUE if c in BUCKET_NAMES.values() else MUTED for c in counts.index]
ax.barh(counts.index, counts.values, color=colors, height=0.6)
for y, v in enumerate(counts.values):
    ax.text(v + 2, y, str(v), va="center", fontsize=9, color="#52514e")
ax.set_title("Hand labels: blue is scored, grey is excluded")
ax.grid(axis="y", visible=False); ax.set_xlabel("Points")
plt.show()

In [ ]:
labels[~is_valid_label][["point_id", "worldcover_class_name", "agreed_label", "confidence", "notes"]]

The second exclusion is NoData: a labelled point that lands on a pixel the classifier did not fill (value 0), or
outside the raster. This is the same sampling and the same test as `evaluate_classifier`:

In [ ]:
valid = labels[is_valid_label]
rasters = {name: PROCESSED_DIR / "landcover" / f"{name}_landcover.tif" for name in ("rf", "unet", "hybrid")}

rows, nodata_ids = [], {}
for model, path in rasters.items():
    sampled = sample_raster_at_points(path, valid)
    no_data = sampled["pred_bucket"].isna() | (sampled["pred_bucket"] == 0)
    nodata_ids[model] = sampled.loc[no_data, "point_id"].tolist()
    rows.append({"classifier": model, "points": len(labels),
                 "excluded: blank/uncertain": int((~is_valid_label).sum()),
                 "excluded: NoData": int(no_data.sum()), "scored": int((~no_data).sum())})
funnel = pd.DataFrame(rows).set_index("classifier")

saved = pd.read_csv(PROCESSED_DIR / "landcover" / "evaluation" / "comparison_table.csv").set_index("model")["n_scored"]
funnel["n_scored in saved evaluation"] = saved
funnel

The `scored` column reproduces the `n_scored` the evaluator saved, so the exclusion counts above are right. Note the
evaluator only **prints** "Skipping N..." and "Dropping N..." while running; the saved CSVs keep `n_scored` alone, so
the excluded counts are not stored anywhere. That is the one place rule 4 ("record the number excluded") is only
half met.

Where are the excluded points?

In [ ]:
excluded_ids = set(labels.loc[~is_valid_label, "point_id"]) | set(nodata_ids["hybrid"])
excluded = labels[labels["point_id"].isin(excluded_ids)]
kept = labels[~labels["point_id"].isin(excluded_ids)]

fig, ax = plt.subplots(figsize=(8.5, 5))
gdf.plot(ax=ax, color=LAND, edgecolor="white", linewidth=0.3)
ax.scatter(kept["lon"], kept["lat"], s=9, color=BLUE, label=f"scored by the hybrid ({len(kept)})")
ax.scatter(excluded["lon"], excluded["lat"], s=70, facecolors="none", edgecolors=ORANGE, linewidths=1.6,
           label=f"excluded ({len(excluded)}): uncertain or NoData")
for i, r in enumerate(excluded.itertuples()):  # stagger so near-coincident points stay readable
    ax.annotate(r.point_id, (r.lon, r.lat), xytext=(9, 7 - 14 * (i % 2)), textcoords="offset points",
                fontsize=8, color="#52514e")
ax.legend(loc="upper left", frameon=False, fontsize=9)
ax.set_title("Validation points, with the excluded ones ringed"); ax.set_axis_off(); ax.grid(False)
plt.show()